In [ ]:
import numpy as np
import sklearn
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split

# Load data once and convert to float32 to save memory
X = np.load('2208_A[8]_rand_traces_full_algo_IV(FIXED)_Keys(1Lakh).npy')

# Define bit lengths and corresponding y arrays (ensure these are populated)
bit_lengths = {
    'last_1_bit': (100, 500, 1000, 5000, 10000, 50000, 100000),
    'last_2_bit': (100, 500, 1000, 5000, 10000, 50000, 100000),
    'last_4_bit': (100, 500, 1000, 5000, 10000, 50000, 100000),
    'last_8_bit': (100, 500, 1000, 5000, 10000, 50000, 100000),
}

# Placeholder for y arrays
y1 = np.load('A8_last_1_bit_array.npy')
y2 = np.load('A8_last_2_bits_array.npy')
y4 = np.load('A8_last_4_bits_array.npy')
y8 = np.load('A8_last_8_bits_array.npy')

# Dictionary to map bit lengths to corresponding y arrays
y_arrays = {
    'last_1_bit': y1,
    'last_2_bit': y2,
    'last_4_bit': y4,
    'last_8_bit': y8,
}

# Initialize results dictionary
results = {key: {'train_accuracy': [], 'test_accuracy': [], 'trace_count_train': [], 'trace_count_test': []} for key in bit_lengths}

# Function to train and evaluate LDA
def train_and_evaluate(X, y, sizes, key):
    for i in sizes:
        if i > len(X):  # Ensure we don't exceed the number of samples
            continue

        X_subset = X[:i]
        y_subset = y[:i]

        # Split data into training and testing sets
        xtrain, xtest, ytrain, ytest = train_test_split(
            X_subset, y_subset, test_size=0.2, random_state=0
        )

        lda = LDA()
        lda.fit(xtrain, ytrain)

        # Compute accuracies
        train_accuracy = lda.score(xtrain, ytrain)
        test_accuracy = lda.score(xtest, ytest)

        results[key]['train_accuracy'].append(train_accuracy)
        results[key]['test_accuracy'].append(test_accuracy)
        results[key]['trace_count_train'].append(len(xtrain))
        results[key]['trace_count_test'].append(len(xtest))

# Loop through each bit length and corresponding y arrays
for key in bit_lengths.keys():
    y = y_arrays[key]  # Get the corresponding y array
    train_and_evaluate(X, y, bit_lengths[key], key)

# Print all accuracies and trace counts
for key in bit_lengths.keys():
    print(f"\n{key} Train Accuracies: {results[key]['train_accuracy']}")
    print(f"{key} Test Accuracies: {results[key]['test_accuracy']}")
    print(f"{key} Train Trace Counts: {results[key]['trace_count_train']}")
    print(f"{key} Test Trace Counts: {results[key]['trace_count_test']}")